# Machine Learning Cloud Deployment Tutorial Series

Tutor: Daniel Ramandi

# Session 2 : From web request to model prediction

In Sessions 1 and 2, we built the deployment backbone: 
- a model artifact in S3
- all the IAM roles
- a SageMaker endpoint (in the optional part)
- an Elastic Beanstalk app that can call that endpoint

The goal of this session is to make the code path fully concrete: 
- what the Flask app does
- what the SageMaker deployment script does
- how the SageMaker `inference` script is actually used at serving time

By the end, you should be able to modify the web app, swap in a new endpoint, and write your own inference.py for a different model.

Before we start, take a moment to [**login to AWS**](https://ubclthubcourses.awsapps.com/start/#) and start setting up your EB app as you did in session 2.

## Part 1 : The Flask app (`app.py`)

### 1.1 What a Flask app is structurally

A Flask app is usually organized around a central application object, helper functions, and route handlers. 

A **route** binds a URL pattern to a Python function, and Flask calls that function when an HTTP request matches the URL and method (don't worry, keep reading, you'll understand it as we move forward). 

Flask’s `@app.route(...)` decorator is exactly the mechanism that binds a function to a URL, and routes can be limited to specific HTTP methods such as `GET` and `POST`.

In the example `app.py` file that I provided for you, there are three kinds of code:

- global setup and configuration,
- helper functions that do work behind the scenes,
- route functions that define the app’s public behavior.

That is a good default mental model for almost any small Flask application.

### 1.2 App creation and configuration

In [ ]:
app = Flask(__name__)

ENDPOINT_NAME = os.environ["ENDPOINT_NAME"]
AWS_REGION = os.environ.get("AWS_REGION", "ca-central-1")

HONEYPOT_FIELD = "website"

`app = Flask(__name__)` creates the Flask application object. Everything else in the file attaches behavior to that object. 

In Flask, this object is what stores the routing table and configuration for the application. Just always add it at the top and you should be good to go!

Here, `ENDPOINT_NAME` and `AWS_REGION` are deployment-time configuration. The important design choice here is that the endpoint name is not hard-coded into the application logic. Instead, Elastic Beanstalk can inject it through environment variables. That makes the app portable, so the exact same code can point to different SageMaker endpoints in dev, staging, or production.

`HONEYPOT_FIELD = "website"` is not a Flask feature; it is just application logic. The idea is that the HTML form contains a hidden field with that name, and many low-quality bots will fill it even though a human user never sees it. This (or something similar and maybe more advanced) is nice to have here specifically, because bot attacks are going to be very expensive for you!


### 1.3 Helper function: creating the SageMaker Runtime client

In [ ]:
def get_runtime_client():
    return boto3.client("sagemaker-runtime", region_name=AWS_REGION)

This is the line that makes the Flask app an AWS client. The service name is sagemaker-runtime, which is the API used for sending inference requests to a live SageMaker endpoint. This is different from the broader SageMaker control plane used to create models and endpoints. In other words, this client is for using an endpoint, not building one.

A useful mental model is:

- **deployment code** talks to SageMaker to create infrastructure (we will cover this later)
- **runtime code** talks to SageMaker Runtime to make predictions

On Elastic Beanstalk, boto3 usually gets credentials from the EC2 instance profile attached to the environment, so there are no access keys in the source code.

### 1.4 Helper function: sending text to the endpoint

In [ ]:
def predict_message(text: str):
    runtime = get_runtime_client()
    payload = json.dumps({"instances": [text]})

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=payload,
    )

    result = json.loads(response["Body"].read().decode("utf-8"))
    label = result["predicted_labels"][0]
    return label, result

This is the core integration point between **Flask** and **SageMaker**. 

The function builds a JSON payload, calls the endpoint, parses the response, and returns both a user-friendly label and the raw prediction dictionary.

`runtime.invoke_endpoint(...)` is the network call from your EB app to the SageMaker endpoint.

The `ContentType="application/json"` matters because the SageMaker model server uses the request content type to decide how to parse the body. In this setup as you will see later, the inference script explicitly expects JSON.

### 1.5 Why Flask routes matter

In [ ]:
@app.route("/", methods=["GET"])
def home():
    return render_template("index.html")

A route is simply **what function should run for this URL and method?** 

Flask’s route decorator binds a URL rule to a Python function. Here, "/" is the home page and it responds to `GET`, which is the standard method for loading a page.

`render_template("index.html")` tells Flask to render a template. This matters because the template HTML we have is not static. It contains placeholders like `{{ message }}`, `{{ prediction }}`. Those are filled in only when Flask renders the template with context variables. 

So the home route is the simplest possible pattern:

- user opens the page,
- Flask returns the template,
- no prediction has happened yet.



### 1.6 Form submission route

In [ ]:
@app.route("/predict", methods=["POST"])
def predict():
    if request.form.get(HONEYPOT_FIELD):
        return render_template(
            "index.html",
            error="Bot-like submission detected. Please try again."
        )

    message = request.form.get("message", "").strip()

    if not message:
        return render_template(
            "index.html",
            error="Please enter a message."
        )

    try:
        label, raw_result = predict_message(message)
        return render_template(
            "index.html",
            message=message,
            prediction=label,
            raw_result=raw_result
        )
    except Exception as e:
        return render_template(
            "index.html",
            message=message,
            error=str(e)
        )

The HTML and Python are tightly coupled here:

- the form submits to /predict,
- Flask defines a /predict route that accepts POST,
- the form field named message is read with `request.form.get("message", "")`.

**There are three steps inside this route.**

- First, validate the `honeypot` field.
- Second, validate the user’s real input.
- Third, call `predict_message()` and render the result back into the same template.

This **submit form → re-render same page with variables** pattern is very common in Flask. In the template `index.html` I gave you, the following pieces are what make that work:

- `{{ message if message else '' }}` repopulates the textarea,
- `{% if prediction %}` conditionally shows the prediction box,
- `{{ prediction|upper }}` formats the label,
- `{% if error %}` shows validation or runtime errors.

That is the key thing you should understand if you want to build your own pages: **the template is just a view layer over variables produced by the route function.**

### 1.7 API route vs page route

In [ ]:
@app.route("/api/predict", methods=["POST"])
def api_predict():
    try:
        payload = request.get_json(force=True)
        message = payload.get("message", "").strip()

        if not message:
            return jsonify({"error": "Missing 'message'"}), 400

        label, raw_result = predict_message(message)

        return jsonify({
            "message": message,
            "prediction": label,
            "raw_result": raw_result
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

This route is conceptually the same as `/predict`, but it is built for programmatic clients instead of browsers. 

**The browser-facing route returns HTML; this one returns JSON.**

### 1.8 Health route

In [ ]:
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

This is a simple liveness endpoint. 

It answers the question **is the Flask app responding?** 

It does not prove the SageMaker endpoint is healthy. It only proves the web app is up enough to answer an HTTP request.

### 1.9 How students can make their own Flask app

To build your own version, you mainly need to change three things:

- the HTML form fields and the route that reads them
- the payload you send to the backend model
- the variables passed into render_template(...) that are referenced in your HTML.

A good rule of thumb is:

- every form field name in HTML should correspond to how you read it in request.form
- every variable used in HTML should be passed from the route
- every route should have a clear input, a clear output, and a clear HTTP method

**YOU CAN READ MORE AND SEE EXAMPLES IN FLASK DOCS [HERE](https://flask.palletsprojects.com/en/stable/quickstart/).**

## Part 2 : The SageMaker deployment script

### 2.1 What this script is for

The Flask app uses a SageMaker endpoint. The deployment script is what creates that endpoint from a model artifact and a serving script. In SageMaker’s hosting flow, the underlying concepts are **CreateModel**, **CreateEndpointConfig**, and **CreateEndpoint**: 

- first define the model and serving container
- then define the compute resources
- then create the live endpoint.

SageMaker’s Python SDK wraps that process in higher-level model classes and `.deploy()`.

That is why this script feels very different from `app.py`. It is infrastructure orchestration, not request handling. 

Also, more importantly, **YOU RUN THIS SCRIPT ON YOUR CLOUDSHELL, NOT YOUR EB INSTANCE**.

### 2.2 Setup and session objects

In [ ]:
boto_session = boto3.Session()
region = boto_session.region_name
sm_session = sagemaker.Session(boto_session=boto_session)

`boto3.Session()` defines your AWS context: **credentials**, **region**, and **default configuration chain**. `sagemaker.Session(...)` is a higher-level wrapper around that AWS context for SageMaker-specific operations such as **S3 uploads** and **endpoint-related calls**. 

The SageMaker [`Session`](https://sagemaker.readthedocs.io/en/v1.54.0/session.html?utm_source=chatgpt.com) class is explicitly documented as a convenience layer for working with SageMaker jobs, endpoints, and S3-backed resources.

### 2.3 Uploading the model artifact

In [ ]:
model_s3_uri = sm_session.upload_data(
    path="model_build/model.tar.gz",
    bucket=BUCKET,
    key_prefix=PREFIX
)

This uploads the local model tarball to S3 and returns its full `s3://...` URI. SageMaker hosting expects model artifacts to live in S3, and the `model_data` parameter of framework model classes points to the S3 location of a SageMaker model `.tar.gz` file.

**NOTE** : This is the first place you should connect the dots with `inference.py`. The files that model_fn loads later (such as model.joblib and metadata.json) must be inside this tarball in the layout the serving script expects.

### 2.4 Defining the deployable SageMaker model

In [ ]:
sk_model = SKLearnModel(
    model_data=model_s3_uri,
    role=ROLE_ARN,
    entry_point="inference.py",
    source_dir="model_build",
    framework_version="1.4-2",
    py_version="py3",
    sagemaker_session=sm_session
)

**This block does not launch anything yet**. It creates a Python object that describes **how** the model should be served.

The most important parameters are:

- `model_data`: where the artifact tarball is in S3
- `role`: the IAM role SageMaker assumes to access artifacts and other AWS resources
- `entry_point`: the Python file SageMaker should use for hosting logic (your `inference.py` file)
- `source_dir`: the directory that contains that script and any support code
- `framework_version` and `py_version`: which SageMaker-managed scikit-learn container/runtime to use. The SageMaker [docs](https://sagemaker.readthedocs.io/en/v2.38.0/frameworks/sklearn/sagemaker.sklearn.html) list 1.4-2 as a supported scikit-learn container version, and py3 as the supported Python version in this API.

The two most important hosting-specific ideas are `entry_point` and `source_dir`. The SageMaker SDK documentation states that for model hosting, `entry_point` is the Python source file executed as the hosting entry point, and if `source_dir` is specified, the `entry_point` file must be at the root of that directory.

**this is basically where SageMaker is told which serving script to use.**

### 2.5 Deploying the endpoint

In [ ]:
predictor = sk_model.deploy(
    endpoint_name=ENDPOINT_NAME,
    instance_type="ml.m5.large",
    initial_instance_count=1
)

This is the line that turns the configuration into real infrastructure. Under the hood, the flow is conceptually:

- create a SageMaker Model
- create an `EndpointConfig` that specifies instance type/count
- create the `Endpoint` itself

AWS documents CreateEndpointConfig as the place where you identify the model and the resources to provision, and CreateEndpoint as the step that actually launches resources and deploys the model.

The arguments map directly to hosting choices:

- `endpoint_name`: the live endpoint’s name
- `instance_type`: what hardware SageMaker should provision
- `initial_instance_count`: how many copies to start with

That endpoint name is exactly what the Flask app later uses in `invoke_endpoint(...)`.

**NOTE**: `deploy()` returns a predictor object, which is a client-side handle to the endpoint the SDK just created. 

Conceptually, this is a convenience wrapper for making inference requests from Python. 

Your EB app does not use that object directly, but it talks to the same endpoint through the lower-level SageMaker Runtime API.

So there are two levels of access:

- in a notebook: call the returned predictor
- in a web app: call the endpoint with boto3.client("sagemaker-runtime")

## Part 3 : The SageMaker inference script (`inference.py`)


### 3.1 Why this file is just a set of functions

This is the conceptual point people usually find confusing (and by people I mean me!).

**`inference.py` is not a standalone app.** It is a serving hook file. 

The SageMaker scikit-learn model server loads your script and calls specific functions from it at specific stages of model loading and request handling. SageMaker’s documentation explicitly separates model loading from model serving and describes serving as a sequence of:
- input processing
- prediction
- output processing

**where each stage receives the return value of the previous stage.**

So the order is not written by you in `inference.py`. The order is **owned by the SageMaker model server**.

A useful mental model is:

```text
Flask app -> InvokeEndpoint -> SageMaker model server -> your inference.py hooks
```

For this setup, the effective lifecycle is:

```text
endpoint container starts
    -> model_fn(model_dir)

each inference request
    -> input_fn(request_body, request_content_type)
    -> predict_fn(input_data, loaded_model)
    -> output_fn(prediction, accept)
```
AWS’s scikit-learn serving docs describe exactly this pattern [here](https://sagemaker.readthedocs.io/en/v1.72.1/frameworks/sklearn/using_sklearn.html).

That is why the file I gave you contains only functions; because **SageMaker is the orchestrator**.

## Part 4 : The full path, end to end

Putting the three files together, the full request path is:

browser submits form to Flask
    -> Flask reads `request.form["message"]`
    -> Flask calls SageMaker Runtime `invoke_endpoint(...)`
    -> SageMaker endpoint receives JSON payload
    -> `input_fn` parses payload
    -> `predict_fn` runs the model
    -> `output_fn` serializes response
    -> Flask receives JSON response
    -> Flask renders `index.html` with prediction variables

That is the full stack you should leave with in mind. 

The Flask app is the user-facing interface, the deployment script creates the serving infrastructure, and inference.py defines the custom logic the SageMaker model server plugs into.

## Part 5 : Rate Me!

I would really appreciate it if you could take a few minutes to fill in the course survey and convey to the teaching team about this tutorial series. I hope with your feedback we can improve this series further and hopefully add/remove material to make it more useful. **MORE IMPORTANTLY, Please take the time to fill in the TA evaluation form as well!**

[**LINK TO COURSE EVALUATION**](https://canvas.ubc.ca/courses/178670/external_tools/53187)